# FHIR Condition Bronze-to-Silver Transformation

## Purpose

Transform raw FHIR Condition resources from the Bronze layer into a
structured Silver Delta table.

### Source
`health_insurance.bronze.fhir_condition_raw`

### Target
`health_insurance.silver.fhir_condition`

### Responsibilities

- Parse raw FHIR JSON
- Extract Condition identifiers and coding
- Extract Patient and Encounter references
- Standardize clinical and verification status
- Convert timestamps to proper data types
- Preserve source lineage metadata

Formal data-quality enforcement is implemented separately in the
dedicated data-quality stage.



In [0]:

# stage configuration


CATALOG = "health_insurance"

SOURCE_TABLE = f"{CATALOG}.bronze.fhir_condition_raw"
TARGET_TABLE = f"{CATALOG}.silver.fhir_condition"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)

In [0]:

# Loading Bronze FHIR Condition data


condition_bronze_df = spark.table(SOURCE_TABLE)

print(f"Rows: {condition_bronze_df.count():,}")
print(f"Columns: {len(condition_bronze_df.columns)}")

condition_bronze_df.printSchema()

display(condition_bronze_df.limit(5))

In [0]:

# Infering combined FHIR Condition JSON schema


schema_result = spark.sql(f"""
    SELECT schema_of_json_agg(raw_json) AS condition_schema
    FROM {SOURCE_TABLE}
""").first()

condition_schema = schema_result["condition_schema"]

print("FHIR Condition schema:")
print(condition_schema)

In [0]:

# Parsing raw FHIR Condition JSON


from pyspark.sql import functions as F

condition_parsed_df = (
    condition_bronze_df
    .withColumn(
        "condition",
        F.from_json(
            F.col("raw_json"),
            condition_schema
        )
    )
)

condition_parsed_df.select("condition.*").printSchema()

In [0]:

# Extracting core Condition attributes


condition_core_df = (
    condition_parsed_df
    .select(
        F.col("condition.id").alias("condition_id"),

        F.col("condition.clinicalStatus").alias("clinical_status_struct"),

        F.col("condition.verificationStatus").alias("verification_status_struct"),

        F.col("condition.code").alias("condition_code_struct"),

        F.col("condition.subject.reference").alias("patient_reference"),

        F.col("condition.encounter.reference").alias("encounter_reference"),

        F.col("condition.onsetDateTime").alias("onset_datetime_raw"),

        F.col("condition.abatementDateTime").alias("abatement_datetime_raw"),

        F.col("condition.recordedDate").alias("recorded_datetime_raw"),

        "_ingested_at",
        "_source_system",
        "_resource_type"
    )
)

In [0]:

# Extracting Condition status values


condition_status_df = (
    condition_core_df

    .withColumn(
        "clinical_status",
        F.element_at(
            F.col("clinical_status_struct.coding.code"),
            1
        )
    )

    .withColumn(
        "verification_status",
        F.element_at(
            F.col("verification_status_struct.coding.code"),
            1
        )
    )
)

In [0]:

# Extracting Condition coding


condition_coded_df = (
    condition_status_df

    .withColumn(
        "condition_code",
        F.element_at(
            F.col("condition_code_struct.coding.code"),
            1
        )
    )

    .withColumn(
        "condition_name",
        F.element_at(
            F.col("condition_code_struct.coding.display"),
            1
        )
    )

    .withColumn(
        "code_system",
        F.element_at(
            F.col("condition_code_struct.coding.system"),
            1
        )
    )
)

In [0]:

# Parsing FHIR references


condition_refs_df = (
    condition_coded_df

    .withColumn(
        "patient_id",
        F.regexp_extract(
            F.col("patient_reference"),
            r"Patient/(.+)",
            1
        )
    )

    .withColumn(
        "encounter_id",
        F.regexp_extract(
            F.col("encounter_reference"),
            r"Encounter/(.+)",
            1
        )
    )
)

In [0]:

# Converting Condition timestamps


condition_typed_df = (
    condition_refs_df

    .withColumn(
        "onset_datetime",
        F.to_timestamp("onset_datetime_raw")
    )

    .withColumn(
        "abatement_datetime",
        F.to_timestamp("abatement_datetime_raw")
    )

    .withColumn(
        "recorded_datetime",
        F.to_timestamp("recorded_datetime_raw")
    )
)

In [0]:

# Deriving Condition duration


condition_enriched_df = (
    condition_typed_df

    .withColumn(
        "condition_duration_days",
        F.datediff(
            F.to_date("abatement_datetime"),
            F.to_date("onset_datetime")
        )
    )
)

In [0]:

# Standardizing Condition text fields


condition_standardized_df = (
    condition_enriched_df

    .withColumn(
        "clinical_status",
        F.upper(F.trim("clinical_status"))
    )

    .withColumn(
        "verification_status",
        F.upper(F.trim("verification_status"))
    )
)

In [0]:

# Building final Silver Condition dataset


condition_silver_df = (
    condition_standardized_df

    .select(
        "condition_id",
        "patient_id",
        "encounter_id",

        "condition_code",
        "condition_name",
        "code_system",

        "clinical_status",
        "verification_status",

        "onset_datetime",
        "abatement_datetime",
        "recorded_datetime",
        "condition_duration_days",

        "_source_system",
        "_resource_type",
        "_ingested_at"
    )

    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp()
    )
)

In [0]:

# Inspecting Silver Condition result


condition_silver_df.printSchema()

display(
    condition_silver_df.limit(20)
)

In [0]:
#reconciling counts

bronze_count = condition_bronze_df.count()
silver_count = condition_silver_df.count()

print(f"Bronze Conditions: {bronze_count:,}")
print(f"Silver Conditions: {silver_count:,}")
print(f"Difference: {bronze_count - silver_count:,}")

In [0]:

# Persisting FHIR Condition Silver table


(
    condition_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TARGET_TABLE)
)

print(f"Created Silver table: {TARGET_TABLE}")

In [0]:
%sql
--quering the newly formed fhir_condition table


SELECT COUNT(*) AS condition_count
FROM health_insurance.silver.fhir_condition;